# 01. Ingestión, EDA, Limpieza Avanzada y Preparación para Modelado

---

## Objetivos de este Notebook

1. Cargar y auditar `listings.csv.gz`.
2. Eliminar variables identificativas o innecesarias.
3. Limpiar `price`, noches mínimas, baños, dormitorios y métricas agregadas.
4. Crear variables derivadas de `name` y `amenities` sin conservar el texto de `name` en las salidas.
5. Validar nulos, duplicados, tipos, rangos y coordenadas.
6. Generar `listings_cleaned` como dataset intermedio legible.



In [ ]:
import os
import ast
import re
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None) 

RAW_LISTINGS = Path("../data/raw/listings.csv.gz")

PROCESSED_DIR = Path("../data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_LISTINGS = PROCESSED_DIR / "listings_cleaned.parquet"





print("Librerías y rutas de trabajo cargadas correctamente.")


## 1. Carga e Inspección del Dataset Raw (`listings.csv.gz`)
Cargamos el dataset raw y visualizamos su estructura original antes de aplicar cualquier filtro o transformación.


In [ ]:
df_raw = pd.read_csv(RAW_LISTINGS, low_memory=False)
print("======================================================================")
print("DATAFRAME INICIAL (RAW) - ANTES DE CUALQUIER LIMPIEZA O RGPD")
print("======================================================================")
print(f"Dimensiones iniciales: {len(df_raw):,} filas {df_raw.shape[1]} columnas")
display(df_raw.head(5))


## 2. Auditoría y Eliminación de Variables con Alto Porcentaje de Nulos (>50%)
Analizamos el porcentaje de valores faltantes por columna y eliminamos aquellas vacías o con más del 50% de nulos.


In [ ]:
# Auditoría de nulos en df_raw
null_counts = df_raw.isnull().sum()
null_pct = (null_counts / len(df_raw)) * 100
null_summary = pd.DataFrame({"Nulos": null_counts, "Porcentaje": null_pct}).sort_values(by="Porcentaje", ascending=False)

print("Top 15 columnas con mayor porcentaje de nulos en el dataset raw:")
display(null_summary.head(15))

# Eliminación de columnas con >50% de nulos
high_null_cols = null_summary[null_summary["Porcentaje"] > 50.0].index.tolist()
df = df_raw.drop(columns=high_null_cols).copy()
print(f"\nEliminadas {len(high_null_cols)} columnas con >50% de nulos: {high_null_cols}")


## 3. Gobernanza de Datos, RGPD y Privacidad Espacial
Eliminamos metadatos personales identificativos de los anfitriones y preservamos la privacidad espacial (desfase de 0-450m en coordenadas).


In [ ]:
pii_columns = [
    "host_name", "host_about", "host_thumbnail_url", "host_picture_url", 
    "host_url", "picture_url", "listing_url"
]
cols_to_drop = [c for c in pii_columns if c in df.columns]
df.drop(columns=cols_to_drop, inplace=True)
print(f"Eliminadas {len(cols_to_drop)} columnas PII identificativas para cumplimiento RGPD.")
print("Verificado: Preservado el enmascaramiento espacial aleatorio de 0-450m en latitude/longitude.")


## 4. Limpieza de Variables Cuantitativas y Filtros de Negocio

### a) Target (`price`)
Formateo a flotante numérico y filtrado en el rango urbano de Málaga (15€ a 800€/noche).

### b) Corta Estancia (`minimum_nights <= 30`)
Filtrado de exigencias de permanencia mayores a 30 días para ceñir el análisis a VUT turísticas.


In [ ]:
if df["price"].dtype == object:
    df["price"] = df["price"].astype(str).str.replace("$", "", regex=False).str.replace(",", "", regex=False).str.strip()
    df["price"] = pd.to_numeric(df["price"], errors="coerce")

initial_count = len(df)
df = df[(df["price"].notna()) & (df["price"] >= 15.0) & (df["price"] <= 800.0)].copy()
print(f"Registros tras filtro de precio [15€ - 800€]: {len(df):,} (descartados: {initial_count - len(df):,})")

before_min = len(df)
df = df[df["minimum_nights"] <= 30].copy()
print(f"Registros tras filtro corta estancia (minimum_nights <= 30): {len(df):,} (descartados residenciales: {before_min - len(df):,})")


## 5. Extracción de Baños e Imputación de Características Físicas

In [ ]:
def parse_bathrooms(text):
    if pd.isna(text):
        return np.nan
    text = str(text).lower()
    match = re.search(r"(\d+(?:\.\d+)?)", text)
    if match:
        return float(match.group(1))
    if "half" in text:
        return 0.5
    return np.nan

df["bathrooms_num"] = df["bathrooms_text"].apply(parse_bathrooms)
df["bathrooms_num"] = df["bathrooms_num"].fillna(df["bathrooms_num"].median())
df["bedrooms"] = df["bedrooms"].fillna(1)

bool_cols = ["host_is_superhost", "host_has_profile_pic", "host_identity_verified", "has_availability", "instant_bookable"]
for col in bool_cols:
    if col in df.columns:
        df[col] = df[col].map({"t": 1, "f": 0, True: 1, False: 0}).fillna(0).astype(int)

print(f"Imputaciones completadas: Baños (mediana)={df['bathrooms_num'].median()}, Dormitorios (mediana)={df['bedrooms'].median()}")


## 6. Tratamiento de métricas agregadas de reputación

Se utilizan únicamente los recuentos y puntuaciones agregadas que ya están incluidos en `listings.csv.gz`. No se carga ni se procesa el contenido textual de las reseñas.


In [ ]:
df["has_reviews"] = (df["number_of_reviews"] > 0).astype(int)

score_cols = [c for c in df.columns if c.startswith("review_scores_")]
for sc in score_cols:
    df[sc] = df[sc].fillna(df[sc].median())

print(f"Imputadas métricas agregadas de puntuación con la mediana. Alojamientos con reseñas: {df['has_reviews'].sum():,}")


## 7. Depuración de Multicolinealidad y Selección Optimizada de Variables

Basándonos en el análisis de matriz de correlaciones de Pearson, depuramos 4 grupos colineales redundantes:
1. **Disponibilidad:** Eliminamos `availability_60` y `availability_90` por colinealidad extrema con `availability_30` ($r = 0.95$). Conservamos `availability_30` y `availability_365`.
2. **Capacidad:** Eliminamos `beds` por colinealidad con `accommodates` ($r = 0.835$) y `bedrooms` ($r = 0.799$).
3. **Métricas agregadas de puntuación:** Eliminamos `accuracy`, `checkin`, `communication` y `value` ($r > 0.85$ con `review_scores_rating`). Conservamos `rating` (global), `cleanliness` (limpieza) y `location` (ubicación).
4. **Volumen agregado de valoraciones:** Eliminamos `number_of_reviews_l30d` por alta frecuencia de ceros. Conservamos `number_of_reviews` y `number_of_reviews_ltm`.
5. **Redundancias:** Eliminamos `license` (texto ruidoso no numérico) y la columna original de `property_type` (utilizando la agrupada).


In [ ]:
def group_property_type(pt):
    if pd.isna(pt):
        return "Otros"

    pt = str(pt).lower()

    if any(x in pt for x in [
        "rental unit", "condo", "serviced apartment",
        "loft", "apartment",
    ]):
        return "Apartamento"

    if "private room" in pt or "shared room" in pt:
        return "Habitacion"

    if any(x in pt for x in [
        "home", "villa", "townhouse", "cottage", "house",
    ]):
        return "Casa_Villa"

    return "Otros"


df["property_type_group"] = (
    df["property_type"]
    .apply(group_property_type)
)


optimized_columns = [
    # Identificación temporal y localización
    "id", "name", "neighbourhood_cleansed", "latitude", "longitude",

    # Tipo y capacidad
    "room_type", "property_type_group", "accommodates",
    "bedrooms", "bathrooms_num",

    # Variable objetivo y condiciones de alquiler
    "price", "minimum_nights", "maximum_nights",
    "instant_bookable", "amenities",

    # Métricas agregadas y anfitrión
    "host_is_superhost", "host_listings_count",
    "number_of_reviews", "number_of_reviews_ltm", "has_reviews",
    "review_scores_rating", "review_scores_cleanliness",
    "review_scores_location",

    # Disponibilidad
    "availability_30", "availability_365",
]

available_opt = [
    column
    for column in optimized_columns
    if column in df.columns
]

df_opt = df[available_opt].copy()


# ------------------------------------------------------------
# AGRUPACIÓN DE CATEGORÍAS POCO FRECUENTES
# ------------------------------------------------------------

# Room types extremadamente escasos.
rare_room_types = [
    "Hotel room",
    "Shared room",
]

df_opt["room_type_group"] = (
    df_opt["room_type"]
    .where(
        ~df_opt["room_type"].isin(rare_room_types),
        "Otros",
    )
)

# Barrios con menos de 50 alojamientos.
MIN_NEIGHBOURHOOD_COUNT = 50

neighbourhood_counts = (
    df_opt["neighbourhood_cleansed"]
    .value_counts()
)

rare_neighbourhoods = (
    neighbourhood_counts[
        neighbourhood_counts < MIN_NEIGHBOURHOOD_COUNT
    ]
    .index
)

df_opt["neighbourhood_group"] = (
    df_opt["neighbourhood_cleansed"]
    .where(
        ~df_opt["neighbourhood_cleansed"].isin(
            rare_neighbourhoods
        ),
        "Otros_barrios",
    )
)


print(f"- Room types agrupados: {rare_room_types}")
print(
    "- Barrios agrupados por tener menos de "
    f"{MIN_NEIGHBOURHOOD_COUNT} registros: "
    f"{list(rare_neighbourhoods)}"
)

print(
    
    f"{df_opt.shape[0]:,} alojamientos "
    f"{df_opt.shape[1]} columnas."
)
print(
    "El id se mantiene solo durante la limpieza "
    "y no se exportará."
)


## 7.1. Variables derivadas de `name` y `amenities`

Antes de eliminar el texto original de `name`, se extraen algunas características
simples relacionadas con su longitud y con la presencia de términos relevantes.
De forma análoga, a partir de `amenities` se generan el número total de servicios
y distintos indicadores binarios de equipamiento.

Estas transformaciones permiten conservar información potencialmente útil para el
modelado sin utilizar directamente el nombre del alojamiento.

In [ ]:
# ============================================================
# VARIABLES DERIVADAS DE NAME Y AMENITIES
# ============================================================

# ------------------------------------------------------------
# 1. TIPOS BINARIOS
# ------------------------------------------------------------

binary_columns = [
    "instant_bookable",
    "host_is_superhost",
    "has_reviews",
]

for column in binary_columns:
    if column in df_opt.columns:
        df_opt[column] = df_opt[column].astype("int8")


# ------------------------------------------------------------
# 2. LIMPIEZA Y VARIABLES DERIVADAS DE NAME
# ------------------------------------------------------------

df_opt["name"] = (
    df_opt["name"]
    .astype("string")
    .str.strip()
)

df_opt["name_length"] = (
    df_opt["name"]
    .str.len()
    .fillna(0)
    .astype("int16")
)

df_opt["name_word_count"] = (
    df_opt["name"]
    .str.split()
    .str.len()
    .fillna(0)
    .astype("int16")
)

name_keyword_groups = {
    "name_mentions_beach": [
        "playa",
        "beach",
        "sea",
        "mar",
        "seafront",
        "ocean",
    ],
    "name_mentions_center": [
        "centro",
        "center",
        "centre",
        "downtown",
        "historic centre",
        "old town",
    ],
    "name_mentions_luxury": [
        "luxury",
        "lujo",
        "premium",
        "exclusive",
        "deluxe",
    ],
    "name_mentions_view": [
        "view",
        "views",
        "vista",
        "vistas",
        "panoramic",
    ],
    "name_mentions_pool": [
        "pool",
        "piscina",
    ],
}

for new_column, keywords in name_keyword_groups.items():
    pattern = "|".join(
        rf"\b{re.escape(keyword)}\b"
        for keyword in keywords
    )

    df_opt[new_column] = (
        df_opt["name"]
        .str.contains(
            pattern,
            case=False,
            na=False,
            regex=True,
        )
        .astype("int8")
    )


# ------------------------------------------------------------
# 3. CONVERTIR AMENITIES A LISTA
# ------------------------------------------------------------

def parse_amenities(value):
    if pd.isna(value):
        return []

    if isinstance(value, list):
        return value

    try:
        parsed = ast.literal_eval(str(value))
        return parsed if isinstance(parsed, list) else []
    except (ValueError, SyntaxError):
        return []


df_opt["amenities"] = (
    df_opt["amenities"]
    .astype("string")
)

df_opt["amenities_list"] = (
    df_opt["amenities"]
    .apply(parse_amenities)
)

df_opt["amenities_count"] = (
    df_opt["amenities_list"]
    .apply(len)
    .astype("int16")
)


# ------------------------------------------------------------
# 4. SERVICIOS RELEVANTES PARA EL PRECIO
# ------------------------------------------------------------

amenity_groups = {
    "has_wifi": [
        "wifi",
        "internet",
    ],
    "has_kitchen": [
        "kitchen",
    ],
    "has_air_conditioning": [
        "air conditioning",
        "central air conditioning",
    ],
    "has_heating": [
        "heating",
    ],
    "has_parking": [
        "parking",
        "garage",
    ],
    "has_pool": [
        "pool",
    ],
    "has_washer": [
        "washer",
        "washing machine",
    ],
    "has_dryer": [
        "dryer",
    ],
    "has_tv": [
        "tv",
        "hdtv",
    ],
    "has_balcony_or_terrace": [
        "balcony",
        "terrace",
        "patio",
    ],
    "has_sea_view": [
        "sea view",
        "ocean view",
        "beach view",
        "waterfront",
    ],
    "has_workspace": [
        "dedicated workspace",
        "workspace",
    ],
    "has_elevator": [
        "elevator",
        "lift",
    ],
    "has_pets_allowed": [
        "pets allowed",
    ],
    "has_crib": [
        "crib",
        "cot",
    ],
    "has_bbq": [
        "bbq",
        "barbecue",
        "grill",
    ],
    "has_gym": [
        "gym",
        "fitness",
    ],
    "has_hot_tub": [
        "hot tub",
        "jacuzzi",
    ],
    "has_breakfast": [
        "breakfast",
    ],
}


def contains_amenity(amenities, keywords):
    amenities_text = " | ".join(
        str(amenity).lower()
        for amenity in amenities
    )

    return int(
        any(
            keyword.lower() in amenities_text
            for keyword in keywords
        )
    )


for new_column, keywords in amenity_groups.items():
    df_opt[new_column] = (
        df_opt["amenities_list"]
        .apply(
            lambda amenities: contains_amenity(
                amenities,
                keywords,
            )
        )
        .astype("int8")
    )


# ------------------------------------------------------------
# 5. COMPROBACIÓN DEL RESULTADO
# ------------------------------------------------------------

derived_columns = [
    "name_length",
    "name_word_count",
    *name_keyword_groups.keys(),
    "amenities_count",
    *amenity_groups.keys(),
]

derived_columns = [
    column
    for column in derived_columns
    if column in df_opt.columns
]

print(
    f"Creadas {len(derived_columns)} variables derivadas "
    "de name y amenities."
)

display(
    df_opt[
        [
            "name",
            "amenities",
            *derived_columns,
        ]
    ].head(5)
)


## 7.2. Control de calidad final, validación y EDA del dataset limpio

Antes de exportar la salida principal de la Fase 1 se comprueban duplicados, tipos, nulos residuales, rangos lógicos, coordenadas, distribución del precio y relaciones básicas entre variables. Las coordenadas sospechosas se señalan para revisión, pero no se eliminan automáticamente.


In [ ]:
# ============================================================
# 1. DUPLICADOS EN LISTINGS
# ============================================================

rows_before_duplicates = len(df_opt)

def make_hashable(value):
    """Convierte estructuras no hashables para auditorías de calidad."""
    if isinstance(value, list):
        return tuple(make_hashable(item) for item in value)
    if isinstance(value, dict):
        return tuple(
            sorted(
                (key, make_hashable(item))
                for key, item in value.items()
            )
        )
    if isinstance(value, set):
        return tuple(sorted(make_hashable(item) for item in value))
    return value


def safe_nunique(series):
    """Cuenta valores únicos aunque la columna contenga listas."""
    return int(
        series
        .apply(make_hashable)
        .nunique(dropna=False)
    )


duplicate_comparison = df_opt.copy()

for column in duplicate_comparison.columns:
    duplicate_comparison[column] = (
        duplicate_comparison[column]
        .apply(make_hashable)
    )

full_duplicates = int(
    duplicate_comparison
    .duplicated()
    .sum()
)

duplicate_public_ids = int(df_opt["id"].duplicated().sum())

print("CONTROL DE DUPLICADOS EN LISTINGS")
print(f"- Filas completamente duplicadas: {full_duplicates:,}")
print(f"- id públicos duplicados: {duplicate_public_ids:,}")

# Cada id representa un único alojamiento. Si hubiera duplicados, se conserva
# una sola fila por id antes de generar las salidas definitivas.
if duplicate_public_ids > 0:
    df_opt = (
        df_opt
        .drop_duplicates(subset=["id"], keep="first")
        .copy()
    )
    print(
        f"Eliminados {rows_before_duplicates - len(df_opt):,} "
        "alojamientos con id duplicado."
    )

assert df_opt["id"].is_unique, "El id público debe ser único en listings."


# ============================================================
# 2. TABLA DE CALIDAD: TIPOS, NULOS Y VALORES ÚNICOS
# ============================================================

quality_summary = pd.DataFrame({
    "variable": df_opt.columns,
    "tipo": df_opt.dtypes.astype(str).values,
    "nulos": df_opt.isna().sum().values,
    "porcentaje_nulos": (
        df_opt.isna().mean().mul(100).round(2).values
    ),
    "valores_unicos": [
        safe_nunique(df_opt[column])
        for column in df_opt.columns
    ],
}).sort_values(
    ["porcentaje_nulos", "variable"],
    ascending=[False, True],
).reset_index(drop=True)

print("\nRESUMEN DE CALIDAD DEL DATASET LIMPIO")
display(quality_summary)


# ============================================================
# 3. COMPROBACIÓN DE VARIABLES BINARIAS
# ============================================================

binary_columns = [
    "instant_bookable",
    "host_is_superhost",
    "has_reviews",
]

binary_validation = {}

for column in binary_columns:
    if column in df_opt.columns:
        observed_values = set(
            df_opt[column]
            .dropna()
            .astype(int)
            .unique()
            .tolist()
        )
        binary_validation[column] = {
            "valores_observados": sorted(observed_values),
            "es_binaria_0_1": observed_values.issubset({0, 1}),
        }

binary_validation_df = (
    pd.DataFrame(binary_validation)
    .T
    .reset_index(names="variable")
)

print("\nVALIDACIÓN DE VARIABLES BINARIAS")
display(binary_validation_df)


# ============================================================
# 4. VALIDACIÓN DE RANGOS LÓGICOS
# ============================================================

range_checks = {
    "price fuera de [15, 800]": int(
        (~df_opt["price"].between(15, 800)).sum()
    ),
    "accommodates <= 0": int(
        (df_opt["accommodates"] <= 0).sum()
    ),
    "bedrooms < 0": int(
        (df_opt["bedrooms"] < 0).sum()
    ),
    "bathrooms_num < 0": int(
        (df_opt["bathrooms_num"] < 0).sum()
    ),
    "minimum_nights fuera de [1, 30]": int(
        (~df_opt["minimum_nights"].between(1, 30)).sum()
    ),
}

if "availability_30" in df_opt.columns:
    range_checks["availability_30 fuera de [0, 30]"] = int(
        (~df_opt["availability_30"].between(0, 30)).sum()
    )

if "availability_365" in df_opt.columns:
    range_checks["availability_365 fuera de [0, 365]"] = int(
        (~df_opt["availability_365"].between(0, 365)).sum()
    )

range_checks_df = (
    pd.Series(range_checks, name="registros_problematicos")
    .rename_axis("comprobacion")
    .reset_index()
)

print("\nVALIDACIÓN DE RANGOS")
display(range_checks_df)


# ============================================================
# 5. COORDENADAS: MARCAR POSIBLES CASOS FUERA DE MÁLAGA
# ============================================================

coordinate_mask = (
    df_opt["latitude"].between(36.5, 36.9)
    & df_opt["longitude"].between(-4.8, -4.0)
)

coordinate_outliers = df_opt.loc[
    ~coordinate_mask,
    [
                "latitude",
        "longitude",
        "neighbourhood_cleansed",
    ],
].copy()

print(
    "\nCoordenadas potencialmente fuera del entorno de Málaga: "
    f"{len(coordinate_outliers):,}"
)

if not coordinate_outliers.empty:
    display(coordinate_outliers.head(20))


# ============================================================
# 6. DISTRIBUCIÓN FINAL DEL PRECIO
# ============================================================

price_summary = df_opt["price"].describe(
    percentiles=[0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]
)

print("\nRESUMEN ESTADÍSTICO DE PRICE")
display(price_summary.to_frame(name="price"))

plt.figure(figsize=(10, 5))
plt.hist(df_opt["price"].dropna(), bins=50)
plt.title("Distribución del precio tras la limpieza")
plt.xlabel("Precio por noche (€)")
plt.ylabel("Número de alojamientos")
plt.grid(True, alpha=0.3)
plt.show()

# log_price se usa solo para el EDA. No se añade todavía al dataset limpio,
# porque la elección de la transformación del target pertenece al modelado.
log_price_eda = np.log1p(df_opt["price"])

plt.figure(figsize=(10, 5))
plt.hist(log_price_eda.dropna(), bins=50)
plt.title("Distribución de log(1 + price)")
plt.xlabel("log(1 + precio)")
plt.ylabel("Número de alojamientos")
plt.grid(True, alpha=0.3)
plt.show()


# ============================================================
# 7. CORRELACIONES NUMÉRICAS BÁSICAS
# ============================================================

correlation_columns = [
    "price",
    "accommodates",
    "bedrooms",
    "bathrooms_num",
    "minimum_nights",
    "number_of_reviews",
    "number_of_reviews_ltm",
    "review_scores_rating",
    "review_scores_cleanliness",
    "review_scores_location",
    "availability_30",
    "availability_365",
]

correlation_columns = [
    column
    for column in correlation_columns
    if column in df_opt.columns
]

correlation_matrix = (
    df_opt[correlation_columns]
    .corr(numeric_only=True)
    .round(2)
)

print("\nMATRIZ DE CORRELACIÓN NUMÉRICA")
display(correlation_matrix)

price_correlations = (
    correlation_matrix["price"]
    .drop("price")
    .sort_values(key=lambda series: series.abs(), ascending=False)
    .to_frame(name="correlacion_con_price")
)

print("\nCORRELACIONES CON PRICE")
display(price_correlations)


# ============================================================
# 8. PRECIO POR CATEGORÍAS PRINCIPALES
# ============================================================

for category in [
    "room_type",
    "property_type_group",
    "neighbourhood_cleansed",
]:
    if category in df_opt.columns:
        category_price_summary = (
            df_opt
            .groupby(category, dropna=False)["price"]
            .agg(["count", "mean", "median"])
            .sort_values("median", ascending=False)
            .round(2)
        )

        print(f"\nPRECIO POR {category}")
        display(category_price_summary)


# ============================================================
# 9. REGISTRO RESUMIDO DEL PROCESO DE LIMPIEZA
# ============================================================

cleaning_log = pd.DataFrame({
    "etapa": [
        "Dataset raw",
        "Tras limpiar y filtrar price",
        "Tras minimum_nights <= 30",
        "Tras eliminar ids duplicados",
        "Dataset final de listings",
    ],
    "registros": [
        len(df_raw),
        before_min,
        rows_before_duplicates,
        len(df_opt),
        len(df_opt),
    ],
})

cleaning_log["eliminados_desde_etapa_anterior"] = (
    cleaning_log["registros"]
    .shift(1)
    .sub(cleaning_log["registros"])
    .fillna(0)
    .astype(int)
)

cleaning_log["porcentaje_conservado_sobre_raw"] = (
    cleaning_log["registros"]
    .div(len(df_raw))
    .mul(100)
    .round(2)
)

print("\nTRAZABILIDAD DE REGISTROS")
display(cleaning_log)

print(
    "\nControl de calidad final de listings completado. "
    f"Dimensiones finales: {df_opt.shape[0]:,} filas × "
    f"{df_opt.shape[1]} columnas."
)


## 8. Exportación de los datasets de listings


In [ ]:
# ============================================================
# EXPORTACIÓN DEL DATASET LIMPIO
# ============================================================

df_listings_export = (
    df_opt
    .drop(
        columns=[
            "id",
            "name",
        ],
        errors="ignore",
    )
    .copy()
)

df_listings_export.to_parquet(
    OUTPUT_LISTINGS,
    index=False,
)


print(
    "Dataset limpio guardado: "
    f"{df_listings_export.shape[0]:,} filas × "
    f"{df_listings_export.shape[1]} columnas"
)

print(f"Ruta: {OUTPUT_LISTINGS}")

print(
    f"\nPrecio medio final: {df_listings_export['price'].mean():.2f} €"
)
print(
    f"Mediana: {df_listings_export['price'].median():.2f} €"
)

## 9. Cuadro comparativo: dataset inicial frente a dataset final

Comparación visual entre una muestra del archivo original y el dataset limpio de listings.


In [ ]:
print("=" * 70)
print("COMPARACIÓN: DATASET RAW FRENTE A DATASET LIMPIO")
print("=" * 70)

print("\n--- ANTES DE LA LIMPIEZA (RAW) ---")
display(
    df_raw[
        [
            "id",
            "host_name",
            "name",
            "neighbourhood_cleansed",
            "bathrooms_text",
            "price",
            "license",
        ]
    ].head(4)
)

print("\n--- DESPUÉS DE LA LIMPIEZA ---")

comparison_columns = [
    column
    for column in [
        "neighbourhood_cleansed",
        "property_type_group",
        "room_type",
        "accommodates",
        "bedrooms",
        "bathrooms_num",
        "price",
        "name_length",
        "name_word_count",
        "amenities_count",
        "review_scores_rating",
    ]
    if column in df_listings_export.columns
]

display(
    df_listings_export[
        comparison_columns
    ].head(4)
)

print(
    "\nEl id público, host_name, license y el texto "
    "original de name no aparecen en la salida limpia."
)


## 10. Verificación del dataset procesado

Como comprobación final, se revisan la dimensión, los tipos de variables y una muestra
del conjunto exportado que se utilizará como entrada en la siguiente fase.

In [ ]:
# ============================================================
# COMPROBACIÓN DEL ARCHIVO GENERADO
# ============================================================

print("Ruta de salida:")
print(f"- Listings limpio: {OUTPUT_LISTINGS.resolve()}")

df_cleaned_preview = None

if OUTPUT_LISTINGS.exists():
    df_cleaned_preview = pd.read_parquet(OUTPUT_LISTINGS)
else:
    print(
        "\nNo existe listings_cleaned.parquet. "
        "Ejecuta primero la celda de exportación."
    )

if df_cleaned_preview is not None:
    print("\n" + "=" * 70)
    print("MUESTRA DE listings_cleaned.parquet")
    print("=" * 70)

    display(df_cleaned_preview.head(5))

    forbidden_columns = [
        column
        for column in ["id", "name"]
        if column in df_cleaned_preview.columns
    ]

    print("\nCOMPROBACIONES FINALES")
    print(f"- Filas: {len(df_cleaned_preview):,}")
    print(f"- Columnas: {df_cleaned_preview.shape[1]}")
    print(f"- Columnas identificativas no deseadas: {forbidden_columns}")

    assert not forbidden_columns

    print("El dataset limpio ha superado las comprobaciones.")

## 11. Cierre de la fase

Tras la limpieza y el análisis exploratorio se obtiene un conjunto de 8.387
alojamientos preparado para la siguiente etapa del proyecto.

El archivo `listings_cleaned.parquet` conserva las variables necesarias para
continuar con la ingeniería de características, excluyendo identificadores y
campos de texto libre que no se utilizarán en el modelado.

En la siguiente fase se incorporará información geoespacial y se construirán
las variables territoriales que posteriormente utilizará el modelo predictivo.

In [ ]:
print("=" * 72)
print("FASE 1 COMPLETADA: DATASET LIMPIO")
print("=" * 72)

status = "OK" if OUTPUT_LISTINGS.exists() else "PENDIENTE"
print(f"{status:9s} | Listings limpio: {OUTPUT_LISTINGS}")

print(
    "\nSiguiente fase:\n"
    "Ingeniería de características tabulares y geoespaciales."
)